# Azure Synapse Analytics Local Feature Demonstrations
This notebook provides local equivalents of **Azure Synapse SQL** concepts using **PostgreSQL** and **Spark SQL**.
It teaches concepts necessary for Azure Data Engineer certification and interview prep.

## 1. Dedicated SQL Pool (PostgreSQL Schemas)
In Azure Synapse, Dedicated SQL Pools store tables across 60 distributions (Hash, Round Robin, or Replicated). We model these architectures inside our PostgreSQL Data Warehouse using custom schema constraints or tables.

In [ ]:
import psycopg2
import os

db_host = os.environ.get('POSTGRES_HOST', 'postgres-dw')
conn = psycopg2.connect(
    host=db_host,
    database="synapse_dw",
    user="postgres",
    password="password"
)
cursor = conn.cursor()

# Create dedicated schema simulating Synapse dedicated SQL Pool
cursor.execute("CREATE SCHEMA IF NOT EXISTS dedicated_sql_pool;")
conn.commit()
print("✅ Dedicated SQL Pool schema initialized.")

## 2. External Tables (PolyBase/Synapse Serverless)
Azure Synapse reads files directly from ADLS Gen2 storage using External Tables. We demonstrate this using **Spark SQL** pointed to MinIO S3 endpoints. Under the hood, this simulates the Serverless SQL Pool querying parquet/csv files without loading them into database storage.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# 1. Create external table in Spark pointing to S3 (MinIO)
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS demo.ext_customer_parquet (
    customer_id INT,
    name STRING,
    market_segment STRING
)
USING parquet
LOCATION 's3://warehouse/external/customer/'
""")
print("✅ Serverless SQL Pool external table created in Spark.")

## 3. PolyBase Ingestion (PostgreSQL COPY)
PolyBase allows fast bulk ingestion of data from files. In PostgreSQL, we simulate this using the `COPY` statement to perform parallel bulk inserts.

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dedicated_sql_pool.stg_orders (
    order_id INT,
    customer_id INT,
    total_price DECIMAL(10, 2),
    order_date DATE
);
""")
conn.commit()
print("✅ Ingestion staging table created. You can simulate bulk PolyBase COPY by running raw SQL scripts or importing CSVs.")

## 4. Partitioned Tables
Table partitioning improves query performance by pruning partitions. In PostgreSQL we use declarative partitioning by range.

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dedicated_sql_pool.fact_sales (
    sale_id INT,
    amount DECIMAL(10, 2),
    sale_date DATE
) PARTITION BY RANGE (sale_date);

-- Create monthly partitions
CREATE TABLE IF NOT EXISTS dedicated_sql_pool.fact_sales_2026_q1 PARTITION OF dedicated_sql_pool.fact_sales
    FOR VALUES FROM ('2026-01-01') TO ('2026-04-01');

CREATE TABLE IF NOT EXISTS dedicated_sql_pool.fact_sales_2026_q2 PARTITION OF dedicated_sql_pool.fact_sales
    FOR VALUES FROM ('2026-04-01') TO ('2026-07-01');
""")
conn.commit()
print("✅ Partitioned fact tables initialized.")

## 5. Materialized Views
Synapse supports Materialized Views to cache pre-computed joins and aggregates. We configure a Postgres materialized view for daily sales totals.

In [ ]:
cursor.execute("""
CREATE MATERIALIZED VIEW IF NOT EXISTS dedicated_sql_pool.mv_daily_sales AS
SELECT 
    sale_date,
    SUM(amount) as daily_total,
    COUNT(sale_id) as transaction_count
FROM dedicated_sql_pool.fact_sales
GROUP BY sale_date;
""")
conn.commit()
print("✅ Materialized View created.")
cursor.close()
conn.close()